# DataAnalysisAgent — интеллектуальный помощник по анализу данных


## Часть 1: Настройка окружения


In [1]:
# Импорт библиотек и конфигурация
from hello_agents import SimpleAgent, HelloAgentsLLM
from hello_agents.tools import Tool, ToolParameter
from typing import Dict, Any, List
import ast
import os

# Параметры LLM
os.environ["LLM_MODEL_ID"] = "Qwen/Qwen3-8B"
os.environ["LLM_API_KEY"] = "ms-9382e20f-96c2-456a-b609-af5c81201066"
os.environ["LLM_BASE_URL"] = "https://api-inference.modelscope.cn/v1/"
os.environ["LLM_TIMEOUT"] = "60"

print("✅ Библиотеки импортированы, конфигурация завершена")


[output cleared — rerun cell after translation]


## Часть 2: Определение инструментов анализа данных


In [2]:
import json
import pandas as pd
from typing import Dict, Any, List

class DataCleaningTool(Tool):
    """Инструмент очистки данных — очистка табличных данных по заданным правилам"""

    def __init__(self):
        super().__init__(
            name="data_cleaner",
            description="Выполняет очистку переданных табличных данных: удаление пустых значений, фильтрация столбцов и т.д."
        )

    def run(self, parameters: Dict[str, Any]) -> str:
        """
        Выполнение очистки данных
        parameters должен содержать:
          - data_json: str, JSON-строка от excel_reader (обязательно)
          - drop_na: bool, удалять ли строки с пустыми значениями (по умолчанию False)
          - columns_to_keep: List[str], список столбцов для сохранения (опционально)
        """
        data_json = parameters.get("data_json")
        if not data_json:
            return "Ошибка: отсутствуют исходные данные (data_json не может быть пустым)"

        try:
            # Разбор исходных данных
            raw_data = json.loads(data_json)
            records = raw_data.get("完整数据", [])
            if not records:
                return "Предупреждение: исходные данные пусты, очистка невозможна"

            df = pd.DataFrame(records)

            # 1. Фильтрация столбцов
            columns_to_keep = parameters.get("columns_to_keep")
            if columns_to_keep:
                missing_cols = [col for col in columns_to_keep if col not in df.columns]
                if missing_cols:
                    return f"Ошибка: указанные столбцы не существуют: {missing_cols}"
                df = df[columns_to_keep]


            # 2. Удаление строк с пустыми значениями
            if parameters.get("drop_na", False):
                original_len = len(df)
                df = df.dropna()
                dropped = original_len - len(df)
                if dropped > 0:
                    pass
            
            df = df.fillna(0)
            # Формирование результата после очистки
            cleaned_records = df.where(pd.notnull(df), None).to_dict(orient='records')
            result = {
                "clean_data": cleaned_records
            }

            return json.dumps(result, ensure_ascii=False, indent=2)

        except json.JSONDecodeError:
            return "Ошибка: data_json не является корректным JSON"
        except Exception as e:
            return f"Ошибка при очистке: {str(e)}"

    def get_parameters(self) -> List[ToolParameter]:
        return [
            ToolParameter(
                name="data_json",
                type="string",
                description="JSON-строка исходных данных",
                required=True
            ),
            ToolParameter(
                name="drop_na",
                type="boolean",
                description="Удалять ли строки с пустыми значениями",
                required=False
            ),
            ToolParameter(
                name="columns_to_keep",
                type="array",
                description="Список имён столбцов для сохранения",
                required=False
            ),
        ]

print("✅ DataCleaningTool определён")


[output cleared — rerun cell after translation]


In [3]:
class DataStatisticsTool(Tool):
    """Инструмент статистики — описательный статистический анализ"""

    def __init__(self):
        super().__init__(
            name="data_statistics",
            description="Описательная статистика: среднее, медиана, стандартное отклонение и т.д."
        )

    def run(self, parameters: Dict[str, Any]) -> str:
        data_json = parameters.get("data_json")
        if not data_json:
            return "Ошибка: отсутствуют данные"

        try:
            raw_data = json.loads(data_json)
            records = raw_data.get("clean_data", [])
            df = pd.DataFrame(records)
            
            # Статистика числовых столбцов
            numeric_stats = {}
            for col in df.select_dtypes(include=[np.number]).columns:
                numeric_stats[col] = {
                    "count": int(df[col].count()),
                    "mean": float(df[col].mean()),
                    "median": float(df[col].median()),
                    "std": float(df[col].std()),
                    "min": float(df[col].min()),
                    "max": float(df[col].max()),
                    "q25": float(df[col].quantile(0.25)),
                    "q75": float(df[col].quantile(0.75))
                }
            
            # Статистика категориальных столбцов
            categorical_stats = {}
            for col in df.select_dtypes(include=['object']).columns:
                value_counts = df[col].value_counts().head(10).to_dict()
                categorical_stats[col] = {
                    "unique_count": int(df[col].nunique()),
                    "top_values": value_counts
                }
            
            result = {
                "shape": f"{len(df)} строк, {len(df.columns)} столбцов",
                "numeric_stats": numeric_stats,
                "categorical_stats": categorical_stats,
            }
            
            return json.dumps(result, ensure_ascii=False, indent=2)
            
        except Exception as e:
            return f"Ошибка статистического анализа: {str(e)}"

    def get_parameters(self) -> List[ToolParameter]:
        return [
            ToolParameter(
                name="data_json",
                type="string",
                description="JSON-строка данных",
                required=True
            )
        ]

print("✅ DataStatisticsTool определён")



[output cleared — rerun cell after translation]


## Часть 3: Создание агента


In [4]:
# Создание реестра инструментов и агента
from hello_agents import ToolRegistry

# Реестр инструментов
tool_registry = ToolRegistry()
tool_registry.register_tool(DataCleaningTool())

system_prompt = """Вы — аналитик данных. Ваши задачи:
    1. Использовать инструмент data_cleaner для очистки данных
    2. Использовать инструмент data_statistics для статистики
    3. Выбрать подходящую диаграмму и построить её кодом echarts, например:
    option = {
        xAxis: {
            type: 'category',
            data: ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
        },
        yAxis: {
            type: 'value'
        },
        series: [
            {
            data: [120, 200, 150, 80, 70, 110, 130],
            type: 'bar'
            }
        ]
    };
    3. Не анализируйте код, не выводите html — только код echarts
    4. В конце на основе данных предоставьте подробный аналитический отчёт
    
    Отчёт должен включать:
    - контекст и цели анализа
    - ключевые находки
    - статистические расчёты, выявление трендов, аномалий или сравнительный анализ
    Избегайте субъективных выводов — заключения должны опираться на данные.
    Выводите отчёт в формате Markdown.
    """
# Создание агента
agent = SimpleAgent(
    name="Помощник по анализу данных",
    llm=HelloAgentsLLM(),
    system_prompt=system_prompt,
    tool_registry=tool_registry
)

print("✅ Агент создан")
print(f"✅ Доступные инструменты: {list(tool_registry._tools.keys())}")


[output cleared — rerun cell after translation]


## Часть 4: Чтение примера таблицы данных


In [ ]:
file_path = "./data/simple_data.xls"

try:
    df = pd.read_excel(file_path)
    # ⚠️ Без очистки! Сохраняем исходные NaN (pandas автоматически преобразует пустые ячейки Excel в NaN)
    data_records = df.to_dict(orient='records') 

    # Формат входных данных для DataCleaningTool
    clean_input = {
        "完整数据": data_records
    }
    sample_data = json.dumps(clean_input, ensure_ascii=False, indent=2)

except FileNotFoundError:
    sample_data = json.dumps({"error": f"Файл Excel не найден: {file_path}"}, ensure_ascii=False)
except Exception as e:
    sample_data = json.dumps({"error": f"Ошибка чтения Excel: {str(e)}"}, ensure_ascii=False)

print(sample_data)


[output cleared — rerun cell after translation]


## Часть 5: Выполнение анализа данных


In [6]:
# Выполнение анализа данных
print("=== Начало анализа данных ===")
result = agent.run(f"Построй диаграммы и проведи анализ следующих данных\n\n{sample_data}\n")

print(result)


[output cleared — rerun cell after translation]


## Часть 6: Сохранение отчёта и диаграммы


In [14]:
import re
import os

echarts_match = re.search(r"option\s*=\s*(\{[\s\S]*?\});", result)
if echarts_match:
    echarts_code = echarts_match.group(1)
    print(echarts_code)
else:
    print("Код ECharts не найден")


[output cleared — rerun cell after translation]


In [8]:
report_match = re.search(r"(# Отчёт по анализу данных[\s\S]*)", result)
if not report_match:
    report_match = re.search(r"(#Отчет об анализе данных[\s\S]*)", result)

markdown_report = report_match.group(1).strip()
print("Markdown-отчёт извлечён")


# ==============================
# 3. Сохранение Markdown-отчёта в файл
# ==============================
output_dir = "./output"
os.makedirs(output_dir, exist_ok=True)
md_path = os.path.join(output_dir, "report.md")

with open(md_path, "w", encoding="utf-8") as f:
    f.write(markdown_report)

print(f"\nMarkdown-отчёт сохранён: {md_path}")


[output cleared — rerun cell after translation]


In [13]:
from IPython.display import HTML

html_code = f'''
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Первый пример ECharts</title>
    <!-- Подключение echarts.js -->
    <script src="https://cdn.staticfile.org/echarts/4.3.0/echarts.min.js"></script>
</head>
<body>
    <!-- DOM-элемент для ECharts с заданными размерами -->
    <div id="main" style="width: 600px;height:400px;"></div>
    <script type="text/javascript">
        // Инициализация экземпляра echarts
        var myChart = echarts.init(document.getElementById('main'));
 
        // Конфигурация и данные диаграммы
        var option = {echarts_code}
        
        // Отображение диаграммы
        myChart.setOption(option);
    </script>
</body>
</html>
'''


In [15]:
from IPython.display import IFrame

# Сохранение HTML-кода в файл
with open('./output/echarts.html', 'w', encoding='utf-8') as f:
    f.write(html_code)

